# SAM 2 High-Resolution Point Reinforcement Pipeline

**Version 2.0: High-Res Point Prompting**
- **Point Prompts**: Automatically generates center points for every symbol to anchor the model.
- **High-Res Loss**: Calculates Loss and IoU at the original image resolution (Upsampling predictions).
- **Aggressive Training**: Higher learning rate (5e-5) and unfreezing more encoder layers.
- **Batch Size 1**: Optimized for stability on Kaggle T4.

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git accelerate pycocotools -q

In [ ]:
import os, cv2, json, torch, numpy as np, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from glob import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel, Sam2Model
from pycocotools import mask as mask_util
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Metrics and Preprocessing

In [ ]:
def compute_iou(preds, targets):
    t = (targets > 0.5).float()
    if preds.shape[0] != t.shape[0]: t = t[:preds.shape[0]]
    
    p_upsampled = torch.nn.functional.interpolate(
        preds.unsqueeze(1), size=t.shape[-2:], mode='bilinear', align_corners=False
    ).squeeze(1)
    
    p_binary = (torch.sigmoid(p_upsampled) > 0.5).float()
    intersection = (p_binary * t).sum(dim=(1, 2))
    union = (p_binary + t).clamp(0, 1).sum(dim=(1, 2))
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()

def compute_pixel_accuracy(preds, targets):
    t = (targets > 0.5).float()
    if preds.shape[0] != t.shape[0]: t = t[:preds.shape[0]]
    
    p_upsampled = torch.nn.functional.interpolate(
        preds.unsqueeze(1), size=t.shape[-2:], mode='bilinear', align_corners=False
    ).squeeze(1)
    
    p_binary = (torch.sigmoid(p_upsampled) > 0.5).float()
    return (p_binary == t).float().mean().item()

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.medianBlur(img, 5)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2RGB))

## 2. Dataset Setup

In [ ]:
class HieroglyphsDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.data_dir = data_dir
        self.processor = processor
        self.json_paths = sorted(glob(os.path.join(data_dir, "*.json")))
        
    def __len__(self):
        return len(self.json_paths)
    
    def __getitem__(self, idx):
        json_path = self.json_paths[idx]
        img_path = json_path.replace(".json", ".jpg")
        image = preprocess_image(img_path)
        if image is None: return None
        
        with open(json_path, 'r') as f: data = json.load(f)
        bboxes, gt_masks, input_points = [], [], []
        for ann in data['annotations']:
            if 'segmentation' not in ann: continue
            x, y, w, h = ann['bbox']
            bboxes.append([x, y, x + w, y + h])
            input_points.append([[x + w/2, y + h/2]])
            mask = (mask_util.decode(ann['segmentation']) > 0).astype(np.float32)
            gt_masks.append(mask)
        
        if not bboxes: return None
        MAX_SYMBOLS = 100
        bboxes = bboxes[:MAX_SYMBOLS]
        input_points = input_points[:MAX_SYMBOLS]
        gt_masks = gt_masks[:MAX_SYMBOLS]
        
        inputs = self.processor(image, input_boxes=[bboxes], input_points=[input_points], return_tensors="pt")
        inputs["gt_masks"] = torch.tensor(np.array(gt_masks), dtype=torch.float32)
        return {k: v.squeeze(0) if k != "gt_masks" else v for k, v in inputs.items()}

def custom_collate(batch):
    return [b for b in batch if b is not None]

processor = AutoProcessor.from_pretrained("facebook/sam2-hiera-large")
BASE_PATH = "/kaggle/input/datasets/karimtawfik/hieroglyphics-segmentation-data-sam-2-format/segmentation.v1-2023-03-12-7-33pm.sam2/"

train_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"train", processor), batch_size=1, shuffle=True, collate_fn=custom_collate)
valid_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"valid", processor), batch_size=1, shuffle=False, collate_fn=custom_collate)
test_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"test", processor), batch_size=1, shuffle=False, collate_fn=custom_collate)

## 3. Training Logic (Aggressive High-Res Hybrid)

In [ ]:
model = AutoModel.from_pretrained("facebook/sam2-hiera-large").to(device)
for name, param in model.named_parameters():
    if any(x in name for x in ["mask_decoder", "neck", "vision_encoder.model.layers.3"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
bce_loss = nn.BCEWithLogitsLoss()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        pred = torch.sigmoid(pred).reshape(-1)
        target = target.reshape(-1)
        intersection = (pred * target).sum()
        return 1 - (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)

dice_loss_fn = DiceLoss()

def calc_loss(pred, target):
    t = (target > 0.5).float()
    if t.shape[0] > pred.shape[0]: t = t[:pred.shape[0]]
    
    p_upsampled = torch.nn.functional.interpolate(
        pred.unsqueeze(1), size=t.shape[-2:], mode='bilinear', align_corners=False
    ).squeeze(1)
    
    return 0.2 * bce_loss(p_upsampled, t) + 0.8 * dice_loss_fn(p_upsampled, t)

best_val_loss, patience, p_counter = float('inf'), 3, 0

for epoch in range(10):
    model.train()
    t_losses, t_ious, t_accs = [], [], []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")
    for batch in pbar:
        if not batch: continue
        for item in batch:
            p_val, boxes, points = item["pixel_values"].unsqueeze(0).to(device), item["input_boxes"].unsqueeze(0).to(device), item["input_points"].unsqueeze(0).to(device)
            gt = item["gt_masks"].to(device)
            out = model(pixel_values=p_val, input_boxes=boxes, input_points=points, multimask_output=False)
            pred = out.pred_masks.squeeze(0)
            loss = calc_loss(pred, gt)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            t_losses.append(loss.item())
            t_ious.append(compute_iou(pred.detach(), gt))
            t_accs.append(compute_pixel_accuracy(pred.detach(), gt))
            pbar.set_postfix({"loss": f"{np.mean(t_losses):.4f}", "iou": f"{np.mean(t_ious):.4f}", "acc": f"{np.mean(t_accs):.4f}"})

    model.eval()
    v_losses, v_ious, v_accs = [], [], []
    pbar_v = tqdm(valid_loader, desc=f"Epoch {epoch+1} [VALID]")
    with torch.no_grad():
        for batch in pbar_v:
            if not batch: continue
            for item in batch:
                p_val, boxes, points = item["pixel_values"].unsqueeze(0).to(device), item["input_boxes"].unsqueeze(0).to(device), item["input_points"].unsqueeze(0).to(device)
                gt = item["gt_masks"].to(device)
                out = model(pixel_values=p_val, input_boxes=boxes, input_points=points, multimask_output=False)
                pred = out.pred_masks.squeeze(0)
                v_losses.append(calc_loss(pred, gt).item())
                v_ious.append(compute_iou(pred, gt))
                v_accs.append(compute_pixel_accuracy(pred, gt))
            pbar_v.set_postfix({"v_loss": f"{np.mean(v_losses):.4f}", "v_iou": f"{np.mean(v_ious):.4f}", "v_acc": f"{np.mean(v_accs):.4f}"})

    avg_vl = np.mean(v_losses)
    if avg_vl < best_val_loss:
        best_val_loss, p_counter = avg_vl, 0
        model.save_pretrained("/kaggle/working/best_sam2_model")
        processor.save_pretrained("/kaggle/working/best_sam2_model")
        print("--> Best SAM 2 Model Saved")
    else:
        p_counter += 1
        if p_counter >= patience: break

## 4. Final Evaluation Pass

In [ ]:
print("Evaluating on TEST set...")
best_model = Sam2Model.from_pretrained("/kaggle/working/best_sam2_model").to(device)
test_ious, test_accs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        if not batch: continue
        item = batch[0]
        p_val, boxes, points = item["pixel_values"].unsqueeze(0).to(device), item["input_boxes"].unsqueeze(0).to(device), item["input_points"].unsqueeze(0).to(device)
        out = best_model(pixel_values=p_val, input_boxes=boxes, input_points=points, multimask_output=False)
        pred = out.pred_masks.squeeze(0)
        test_ious.append(compute_iou(pred.cpu(), item["gt_masks"]))
        test_accs.append(compute_pixel_accuracy(pred.cpu(), item["gt_masks"]))
print(f"FINAL RESULTS | Test mIoU: {np.mean(test_ious):.4f} | Test Pixel Accuracy: {np.mean(test_accs):.4f}")